## Libraries

In [1]:
import sys
sys.path.append('/hdd4/giuder/Documents/projects/dime')

import faiss
import numpy as np
import pandas as pd
from src.data_loading import CollectionLoader

In [2]:
# src/dime/report.py
import logging

import pandas as pd

from src.config import RUNS_DIR, COLLECTIONS
from src.data_loading import CollectionLoader
from src.evaluate import evaluate, summary as eval_summary

logger = logging.getLogger(__name__)

DEFAULT_MEASURES = ["nDCG@10", "AP", "R@1000", "RR@10"]

def baseline_report(
    collections: str | list[str] = None,
    measures: list[str] = DEFAULT_MEASURES,
    models: list[str] | None = None,
    float_fmt: str = ".4f",
) -> pd.DataFrame:
    """
    Load all baseline runs, evaluate them, and return a MultiIndex table:

        model          dl19                    dl20
                       nDCG@10   AP            nDCG@10   AP
        ance           0.7105    0.3987        0.6891    0.3712
        contriever     0.7231    0.4102        0.7012    0.3891

    With a single collection, columns are flat (no MultiIndex).

    Args:
        
        : collection name or list (defaults to all in config)
        measures:    metrics to compute, e.g. ["nDCG@10", "AP"]
        models:      optional — restrict to these model names
        float_fmt:   display format for float columns

    Returns:
        DataFrame indexed by model with MultiIndex columns (collection, measure)
        or flat columns if a single collection is passed.

    Usage:
        baseline_report("dl19")
        baseline_report(["dl19", "dl20"])
        baseline_report(["dl19", "dl20"], measures=["nDCG@10", "AP"])
        baseline_report(["dl19", "dl20"], models=["contriever", "ance"])
    """
    if collections is None:
        collections = list(COLLECTIONS.keys())
    if isinstance(collections, str):
        collections = [collections]

    multi = len(collections) > 1

    # model → collection → measure → value
    data: dict[str, dict[str, dict[str, float]]] = {}

    for collection in collections:
        runs_dir = RUNS_DIR / collection
        if not runs_dir.exists():
            logger.warning(f"No runs directory at {runs_dir} — skipping.")
            continue

        qrels = CollectionLoader(collection).qrels()

        for path in sorted(runs_dir.glob("*.tsv")):
            if "__" in path.stem:
                continue                          # skip DIME sweep files

            model_name = path.stem
            if models and model_name not in models:
                continue

            logger.info(f"Evaluating {collection}/{path.name} ...")
            run = pd.read_csv(
                path, sep="\t", header=None,
                names=["query_id", "Q0", "doc_id", "rank", "score", "run"],
                dtype={"query_id": str, "doc_id": str},
            )

            summ = eval_summary(evaluate(run, qrels, measures))

            data.setdefault(model_name, {}).setdefault(collection, {})
            for _, row in summ.iterrows():
                if row["measure"] in measures:
                    data[model_name][collection][row["measure"]] = row["mean"]

    if not data:
        raise ValueError(f"No baseline runs found for collections={collections}.")

    # ── build MultiIndex DataFrame ─────────────────────────────────────────────
    if multi:
        columns = pd.MultiIndex.from_product(
            [collections, measures],
            names=["collection", "measure"],
        )
        df = pd.DataFrame(
            index=sorted(data.keys()),
            columns=columns,
            dtype=float,
        )
        df.index.name = "model"
        for model, coll_data in data.items():
            for collection, measure_data in coll_data.items():
                for measure, value in measure_data.items():
                    if (collection, measure) in df.columns:
                        df.loc[model, (collection, measure)] = value
    else:
        collection = collections[0]
        df = pd.DataFrame(
            {
                measure: {
                    model: data[model].get(collection, {}).get(measure, float("nan"))
                    for model in sorted(data.keys())
                }
                for measure in measures
            }
        )
        df.index.name = "model"
        df.columns.name = "measure"

    print(df.to_string(float_format=f"{{:{float_fmt}}}".format))
    print()

    return df

In [4]:
def sanity_check(
    collections: str | list[str] = None,
    models: list[str] | None = None,
) -> pd.DataFrame:
    """
    For each (collection, model), check that every query_id in the run
    has at least one relevance judgment in qrels.

    Prints a summary and returns a DataFrame with one row per
    (collection, model) showing counts and any missing query ids.

    Usage:
        sanity_check("dl19")
        sanity_check(["dl19", "dl20"])
        sanity_check(["dl19", "dl20"], models=["contriever"])
    """
    if collections is None:
        collections = list(COLLECTIONS.keys())
    if isinstance(collections, str):
        collections = [collections]

    rows = []
    all_ok = True

    for collection in collections:
        runs_dir = RUNS_DIR / collection
        if not runs_dir.exists():
            logger.warning(f"No runs directory at {runs_dir} — skipping.")
            continue

        qrels      = CollectionLoader(collection).qrels()
        qrels_qids = set(qrels["query_id"].astype(str).unique())

        for path in sorted(runs_dir.glob("*.tsv")):
            if "__" in path.stem:
                continue

            model_name = path.stem
            if models and model_name not in models:
                continue

            run = pd.read_csv(
                path, sep="\t", header=None,
                names=["query_id", "Q0", "doc_id", "rank", "score", "run"],
                dtype={"query_id": str, "doc_id": str},
            )

            run_qids  = set(run["query_id"].unique())
            missing   = sorted(run_qids - qrels_qids)
            ok        = len(missing) == 0
            if not ok:
                all_ok = False

            rows.append({
                "collection":   collection,
                "model":        model_name,
                "run_queries":  len(run_qids),
                "qrel_queries": len(qrels_qids),
                "missing":      len(missing),
                "ok":           "✓" if ok else "✗",
                "missing_ids":  missing,
            })

    if not rows:
        raise ValueError(f"No baseline runs found for collections={collections}.")

    df = pd.DataFrame(rows).set_index(["collection", "model"])

    print(df[["run_queries", "qrel_queries", "missing", "ok"]].to_string())
    print()
    if all_ok:
        print("All queries have relevance judgments ✓")
    else:
        print("Missing query ids:")
        for (collection, model), row in df.iterrows():
            if row["missing_ids"]:
                print(f"  {collection}/{model}: {row['missing_ids']}")
    print()

    return df

In [3]:
df = baseline_report(["dl19", "dl20", "dlhard"], measures=["nDCG@10", "AP"])

collection    dl19           dl20         dlhard       
measure    nDCG@10     AP nDCG@10     AP nDCG@10     AP
model                                                  
ance        0.6422 0.3610  0.6453 0.3918  0.3270 0.1808
contriever  0.6774 0.4935  0.6693 0.4793  0.3813 0.2446
tasb        0.7169 0.4761  0.6849 0.4757  0.3762 0.2387



In [10]:
path = '/hdd4/giuder/Documents/projects/dime/data/runs/dl19/ance.tsv'
run = pd.read_csv(
    path, sep="\t", header=None,
    names=["query_id", "Q0", "doc_id", "rank", "score", "run"],
    dtype={"query_id": str, "doc_id": str},
)

queries      = CollectionLoader('dl19').qrels()
query_ids = queries["query_id"].tolist()

In [11]:
topk_run = (
    run[
        run["query_id"].isin(query_ids) &
        (run["rank"] < 5)
    ]
    .copy()
)

In [12]:
topk_run

,query_id,Q0,doc_id,rank,score,run
0,156493,Q0,2928707,0,712.46430,ance
1,156493,Q0,1960257,1,712.25040,ance
2,156493,Q0,8182162,2,711.54930,ance
3,156493,Q0,2612493,3,711.49460,ance
4,156493,Q0,8182159,4,711.31830,ance
...,...,...,...,...,...,...
42000,146187,Q0,8434619,0,716.62933,ance
42001,146187,Q0,8434623,1,716.34705,ance
42002,146187,Q0,8434625,2,716.12710,ance
42003,146187,Q0,8434624,3,714.81550,ance


In [13]:
all_dids   = topk_run["doc_id"].tolist()
all_dids

['2928707',
 '1960257',
 '8182162',
 '2612493',
 '8182159',
 '4511137',
 '3838638',
 '1901879',
 '8160520',
 '8160519',
 '7952971',
 '4337526',
 '4337527',
 '3633196',
 '6093907',
 '8612903',
 '8612909',
 '8612910',
 '799647',
 '8612906',
 '533011',
 '8737051',
 '8594272',
 '8594271',
 '1231677',
 '8833194',
 '6776717',
 '7674859',
 '1509940',
 '5339620',
 '2210780',
 '3830857',
 '8252763',
 '1381477',
 '8731617',
 '4922619',
 '3023123',
 '1824485',
 '190806',
 '8255706',
 '8617271',
 '8306451',
 '467601',
 '269428',
 '5466810',
 '8760867',
 '8760866',
 '8760871',
 '2868740',
 '5099520',
 '3538160',
 '82107',
 '8402971',
 '82108',
 '8029524',
 '528372',
 '1804644',
 '4834547',
 '7326934',
 '3338827',
 '7125980',
 '8052624',
 '5189913',
 '188213',
 '182369',
 '5653659',
 '2978866',
 '6898289',
 '2978864',
 '8785371',
 '6122722',
 '8446501',
 '8446505',
 '4458509',
 '8093931',
 '7407803',
 '985488',
 '8283532',
 '7407812',
 '332401',
 '5179251',
 '5475308',
 '8390518',
 '8383461',
 '1277

In [14]:
topk_run = topk_run.reset_index(drop=True)
topk_run["_emb_row"] = np.arange(len(topk_run))

In [15]:
topk_run

,query_id,Q0,doc_id,rank,score,run,_emb_row
0,156493,Q0,2928707,0,712.46430,ance,0
1,156493,Q0,1960257,1,712.25040,ance,1
2,156493,Q0,8182162,2,711.54930,ance,2
3,156493,Q0,2612493,3,711.49460,ance,3
4,156493,Q0,8182159,4,711.31830,ance,4
...,...,...,...,...,...,...,...
210,146187,Q0,8434619,0,716.62933,ance,210
211,146187,Q0,8434623,1,716.34705,ance,211
212,146187,Q0,8434625,2,716.12710,ance,212
213,146187,Q0,8434624,3,714.81550,ance,213
